In [ ]:
import csv
import math
import re
import unicodedata
from pathlib import Path
from typing import NamedTuple

In [ ]:
class Documento(NamedTuple):
    id: int
    titulo: str
    categoria: str
    texto: str

class Resultado(NamedTuple):
    doc: Documento
    score: float

In [ ]:
RUTA_CSV = Path("base_planetas.csv")
def cargar_csv(ruta: Path) -> list["Documento"]:
    docs = []
    with open(ruta, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        columnas_requeridas = {"id", "titulo", "categoria", "texto"}

        for fila in reader:
            if not fila["texto"].strip():   # omitir filas vacías
                continue
            docs.append(Documento(
                id=int(fila["id"]),
                titulo=fila["titulo"].strip(),
                categoria=fila["categoria"].strip(),
                texto=fila["texto"].strip(),
            ))
    return docs


BASE_CONOCIMIENTO = cargar_csv(RUTA_CSV)
categorias = sorted({d.categoria for d in BASE_CONOCIMIENTO})


In [ ]:
def tokenizar(texto: str) -> list[str]: #Normaliza, elimina acentos y devuelve tokens de al menos 3 caracteres
    normalizado = unicodedata.normalize("NFD", texto.lower())
    sin_tildes = "".join(c for c in normalizado if unicodedata.category(c) != "Mn")
    sin_puntuacion = re.sub(r"[^a-z0-9\s]", " ", sin_tildes)
    return [w for w in sin_puntuacion.split() if len(w) > 2]

print(tokenizar("¿CómO funcionA la Propulsión iónicA?"))#prueba

['como', 'funciona', 'propulsion', 'ionica']


In [ ]:
class IndiceRAG:
    def __init__(self, documentos: list[Documento]):
        self.documentos = documentos
        self.vocabulario: list[str] = []
        self.idf: dict[str, float] = {}
        self.vectores: list[list[float]] = []
        self._construir()

    def _construir(self):
        tokens_por_doc = [
            tokenizar(f"{d.titulo} {d.categoria} {d.texto}")
            for d in self.documentos
        ]
        vocab_set: set[str] = set()
        for tokens in tokens_por_doc:
            vocab_set.update(tokens)
        self.vocabulario = sorted(vocab_set)

        n = len(self.documentos)
        for palabra in self.vocabulario:
            df = sum(1 for tokens in tokens_por_doc if palabra in tokens)
            self.idf[palabra] = math.log((n + 1) / (df + 1)) + 1

        self.vectores = [self._vectorizar(t) for t in tokens_por_doc]

    def _vectorizar(self, tokens: list[str]) -> list[float]:
        tf: dict[str, float] = {}
        total = len(tokens) or 1
        for t in tokens:
            tf[t] = tf.get(t, 0) + 1
        return [(tf.get(w, 0) / total) * self.idf.get(w, 1) for w in self.vocabulario]

    @staticmethod
    def _coseno(a: list[float], b: list[float]) -> float:
        dot   = sum(x * y for x, y in zip(a, b))
        norm_a = math.sqrt(sum(x * x for x in a))
        norm_b = math.sqrt(sum(y * y for y in b))
        return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

    def buscar(self, consulta: str, top_k: int = 3) -> list[Resultado]:
        vec_q = self._vectorizar(tokenizar(consulta))
        puntuaciones = [
            Resultado(doc=self.documentos[i], score=self._coseno(vec_q, self.vectores[i]))
            for i in range(len(self.documentos))
        ]
        return sorted(puntuaciones, key=lambda r: r.score, reverse=True)[:top_k]


# Construir el índice
indice = IndiceRAG(BASE_CONOCIMIENTO)

In [ ]:
def generar_respuesta(consulta: str, resultados: list[Resultado]) -> str: #Combina los fragmentos más relevantes en una respuesta coherente
    if not resultados or resultados[0].score < 0.01:
        return "No encontré información relevante para esa consulta."

    mejor = resultados[0].doc
    q_norm = unicodedata.normalize("NFD", consulta.lower())
    q_norm = "".join(c for c in q_norm if unicodedata.category(c) != "Mn")

    if any(p in q_norm for p in ["que es", "que son", "que significa"]):
        intro = f"{mejor.titulo}: "
    elif any(p in q_norm for p in ["como funciona", "como se forma", "como"]):
        intro = "En cuanto al funcionamiento: "
    elif any(p in q_norm for p in ["hay ", "existe", "tienen"]):
        intro = "Sobre ese fenómeno: "
    elif any(p in q_norm for p in ["donde", "distancia"]):
        intro = "Respecto a la ubicación: "
    else:
        intro = ""

    oraciones = re.split(r"(?<=[.!?])\s+", mejor.texto.strip())
    respuesta = " ".join(oraciones[:2])

    if len(resultados) > 1 and resultados[1].score > 0.05:
        oraciones2 = re.split(r"(?<=[.!?])\s+", resultados[1].doc.texto.strip())
        if oraciones2 and oraciones2[0][:20].lower() not in respuesta.lower():
            frase = oraciones2[0]
            respuesta += " Además, " + frase[0].lower() + frase[1:]

    return intro + respuesta

In [ ]:
def consultar(pregunta: str, top_k: int = 3):# Pipeline RAG completo: recupera documentos y genera respuesta.
    print(f"  Consulta: {pregunta}")

    resultados = indice.buscar(pregunta, top_k=top_k)
    respuesta = generar_respuesta(pregunta, resultados)
    return respuesta

In [ ]:
consultar("¿Cómo se forma un agujero negro?")

  Consulta: ¿Cómo se forma un agujero negro?


'En cuanto al funcionamiento: Stephen Hawking predijo en 1974 que los agujeros negros emiten radiación térmica cuántica y se evaporan gradualmente. La temperatura de Hawking es inversamente proporcional a la masa del agujero negro por lo que los más pequeños son más calientes.'

In [ ]:
pregunta = "¿Qué es el telescopio James Webb?"

consultar(pregunta)

  Consulta: ¿Qué es el telescopio James Webb?


'Telescopio James Webb: El JWST opera en infrarrojo desde el punto Lagrange L2 a 1.5 millones de km. Su espejo de 6.5 metros con 18 segmentos hexagonales de berilio dorado capta luz de las primeras galaxias a redshift z mayor a 13. Además, la materia oscura constituye el 27% del universo pero no emite ni absorbe luz electromagnética.'

In [ ]:
mi_pregunta= "¿Donde está la tierra?"
consultar(mi_pregunta)

  Consulta: ¿Donde está la tierra?


'Respecto a la ubicación: El viento solar es un flujo continuo de partículas cargadas principalmente protones y electrones que el sol expulsa a 400-800 km/s. La heliosfera es la burbuja magnética formada por el viento solar que protege al sistema solar de los rayos cósmicos interestelares. Además, la sonda Parker Solar Probe lanzada en 2018 se acercará a solo 6 millones de km del sol en 2025 viajando a 690000 km/h.'

In [ ]:
pregunta1 ="¿Es posible viajar a marte?"
consultar(pregunta1)

  Consulta: ¿Es posible viajar a marte?


'Terraformar Marte requeriría calentar el planeta liberando CO2 congelado crear un campo magnético artificial y construir ecosistemas biológicos cerrados. El proceso tardaría siglos o milenios. Además, marte alberga el volcán más alto del sistema solar el Olimpo Mons con 21 km de altura y 600 km de diámetro.'

In [ ]:
pregunta2="¿Hay vida en marte?"
consultar(pregunta2)

  Consulta: ¿Hay vida en marte?


'Sobre ese fenómeno: La panspermia propone que la vida o sus precursores pueden viajar entre planetas protegidos en el interior de meteoritos. Bacterias como Deinococcus radiodurans sobreviven dosis de radiación miles de veces letales para humanos siendo candidatas a viajes interplanetarios. Además, terraformar Marte requeriría calentar el planeta liberando CO2 congelado crear un campo magnético artificial y construir ecosistemas biológicos cerrados.'

In [ ]:
pregunta3="¿Qué es una nebulosa?"
consultar(pregunta3)

  Consulta: ¿Qué es una nebulosa?


'Gigantes rojas y enanas blancas: Cuando el sol agote su hidrógeno en unos 5000 millones de años se expandirá hasta 200 veces su tamaño actual engullendo Mercurio y Venus. Las capas externas se dispersarán formando una nebulosa planetaria mientras el núcleo quedará como enana blanca de carbono y oxígeno. Además, las nebulosas de emisión como la nebulosa de Orión son nubes de gas ionizado por la radiación ultravioleta de estrellas jóvenes masivas.'

In [ ]:
pregunta4="Dime que es el sistema solar"
consultar(pregunta4)

  Consulta: Dime que es el sistema solar


'Formación del sistema solar: El sistema solar se formó hace 4600 millones de años a partir de una nube de gas y polvo que colapsó gravitacionalmente. La nebulosa solar giratoria se aplanó en un disco protoplanetario donde los planetas se acretaron por colisiones sucesivas. Además, el sistema solar interior comprende Mercurio Venus la Tierra y Marte junto con el cinturón de asteroides.'

In [ ]:
consultar("Describe a venus")

  Consulta: Describe a venus


'Venus es el planeta más caliente del sistema solar con 465 grados Celsius en superficie a pesar de estar más lejos del sol que Mercurio. Su efecto invernadero extremo se debe a una atmósfera de 92 atmósferas de presión compuesta principalmente de CO2 con nubes de ácido sulfúrico. Además, el sistema solar interior comprende Mercurio Venus la Tierra y Marte junto con el cinturón de asteroides.'